# Prepare 2026 Test Set

Notebook này chuẩn hóa raw test set 2026 để có schema giống `Hanoi_final.csv` trước khi đưa vào model.


In [1]:
from pathlib import Path
import sys
import hashlib

import pandas as pd


In [2]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data/raw/test/Hanoi_test_set.csv").exists() and (PROJECT_ROOT.parent / "data/raw/test/Hanoi_test_set.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
from src.weather_data import validate_hourly_frame

RAW_TEST_PATH = PROJECT_ROOT / "data/raw/test/Hanoi_test_set.csv"
TRAIN_FINAL_PATH = PROJECT_ROOT / "data/processed/Hanoi_final.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/Hanoi_test_final.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print(RAW_TEST_PATH)
print(OUTPUT_PATH)


/Users/leducanhtuan/Downloads/Weather-Forecasting-ystem-main/data/raw/test/Hanoi_test_set.csv
/Users/leducanhtuan/Downloads/Weather-Forecasting-ystem-main/data/processed/Hanoi_test_final.csv


## 1. Đọc raw test set


In [3]:
df = pd.read_csv(RAW_TEST_PATH)
df["time"] = pd.to_datetime(df["time"], errors="raise")
df = df.drop_duplicates().sort_values("time").reset_index(drop=True)
for column in df.columns:
    if column not in ["time", "city"]:
        df[column] = pd.to_numeric(df[column], errors="raise")
validate_hourly_frame(df)
print(f"Rows: {len(df):,}; columns: {len(df.columns):,}")
print(f"Time: {df['time'].min()} -> {df['time'].max()}")
df.head()


Rows: 6,528; columns: 34
Time: 2026-01-01 00:00:00 -> 2026-09-29 23:00:00


,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,precipitation,rain,snowfall,pressure_msl,surface_pressure,...,soil_temperature_100_to_255cm,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_moisture_28_to_100cm,soil_moisture_100_to_255cm,weather_code,is_day,city,latitude,longitude
0,2026-01-01 00:00:00,20.5,92,19.1,22.7,0.0,0.0,0.0,1013.0,1010.8,...,23.3,0.299,0.302,0.303,0.387,3,0,Hanoi,21.0285,105.8542
1,2026-01-01 01:00:00,20.7,90,19.0,22.9,0.2,0.2,0.0,1013.0,1010.8,...,23.3,0.300,0.302,0.303,0.387,51,0,Hanoi,21.0285,105.8542
2,2026-01-01 02:00:00,20.7,90,19.0,22.8,0.2,0.2,0.0,1012.8,1010.6,...,23.3,0.301,0.302,0.303,0.387,51,0,Hanoi,21.0285,105.8542
3,2026-01-01 03:00:00,20.4,91,18.9,22.7,0.4,0.4,0.0,1012.6,1010.4,...,23.3,0.307,0.302,0.303,0.387,51,0,Hanoi,21.0285,105.8542
4,2026-01-01 04:00:00,20.1,92,18.7,22.3,0.4,0.4,0.0,1012.8,1010.6,...,23.3,0.310,0.302,0.303,0.387,51,0,Hanoi,21.0285,105.8542


## 2. Giữ các biến gốc để khớp train set


In [4]:
train_times = pd.read_csv(TRAIN_FINAL_PATH, usecols=["time"], parse_dates=["time"])["time"]
if df["time"].min() != train_times.max() + pd.Timedelta(hours=1):
    raise ValueError("External test must start exactly one hour after training data ends.")
print("Verified: test follows train with no overlap or missing hours.")


Verified: test follows train with no overlap or missing hours.


## 3. Ép schema giống `Hanoi_final.csv`


In [5]:
train_columns = pd.read_csv(TRAIN_FINAL_PATH, nrows=0).columns.tolist()
missing_columns = [column for column in train_columns if column not in df.columns]
extra_columns = [column for column in df.columns if column not in train_columns]

print("Missing columns compared with train schema:", missing_columns)
print("Extra columns dropped:", extra_columns)

if missing_columns:
    raise ValueError(f"Test set is missing required columns: {missing_columns}")

df = df[train_columns]
print(f"Final columns: {len(df.columns):,}")
df.head()


Missing columns compared with train schema: []
Extra columns dropped: ['relative_humidity_2m', 'precipitation', 'rain', 'snowfall', 'pressure_msl', 'surface_pressure', 'cloud_cover_low', 'cloud_cover_mid', 'cloud_cover_high', 'wind_speed_10m', 'wind_speed_100m', 'wind_direction_10m', 'wind_direction_100m', 'wind_gusts_10m', 'soil_temperature_28_to_100cm', 'soil_temperature_100_to_255cm', 'soil_moisture_0_to_7cm', 'soil_moisture_7_to_28cm', 'soil_moisture_28_to_100cm', 'soil_moisture_100_to_255cm', 'weather_code']
Final columns: 13


,time,city,latitude,longitude,temperature_2m,apparent_temperature,et0_fao_evapotranspiration,vapour_pressure_deficit,soil_temperature_0_to_7cm,soil_temperature_7_to_28cm,dew_point_2m,is_day,cloud_cover
0,2026-01-01 00:00:00,Hanoi,21.0285,105.8542,20.5,22.7,0.01,0.20,21.5,21.2,19.1,0,99
1,2026-01-01 01:00:00,Hanoi,21.0285,105.8542,20.7,22.9,0.01,0.24,21.7,21.1,19.0,0,100
2,2026-01-01 02:00:00,Hanoi,21.0285,105.8542,20.7,22.8,0.01,0.24,21.8,21.0,19.0,0,100
3,2026-01-01 03:00:00,Hanoi,21.0285,105.8542,20.4,22.7,0.01,0.22,21.5,21.0,18.9,0,98
4,2026-01-01 04:00:00,Hanoi,21.0285,105.8542,20.1,22.3,0.00,0.20,21.3,20.9,18.7,0,67


## 4. Kiểm tra và lưu


In [6]:
time_diff = df["time"].diff().dropna()
quality = pd.DataFrame({
    "metric": ["rows", "columns", "missing_cells", "duplicate_timestamps", "non_hourly_intervals", "start_time", "end_time"],
    "value": [
        len(df),
        len(df.columns),
        int(df.isna().sum().sum()),
        int(df["time"].duplicated().sum()),
        int((time_diff != pd.Timedelta(hours=1)).sum()),
        df["time"].min(),
        df["time"].max(),
    ],
})
quality


,metric,value
0,rows,6528
1,columns,13
2,missing_cells,0
3,duplicate_timestamps,0
4,non_hourly_intervals,0
5,start_time,2026-01-01 00:00:00
6,end_time,2026-09-29 23:00:00


In [7]:
validate_hourly_frame(df, train_columns)

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")


Saved: /Users/leducanhtuan/Downloads/Weather-Forecasting-ystem-main/data/processed/Hanoi_test_final.csv
Rows: 6,528
Columns: 13
